# CIS 536/736: Computer Graphics
## Practice Lab 19: Rigging, Forward Kinematics, and LBS

**Objective:** In this practice lab, you will implement a 2D mathematical foundation for skeletal animation. You will build a Forward Kinematics (FK) solver to compute global bone transformations from local hierarchical transforms, and apply Linear Blend Skinning (LBS) to deform a mesh vertex based on bone weights.

*Note: While MP 6 will require OpenUSD and visual graph rigging, understanding the underlying transformation matrices is a mandatory prerequisite.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Utility: Create a 2D homogeneous transformation matrix
def transform_2d(tx, ty, theta_deg):
    theta = np.radians(theta_deg)
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [c, -s, tx],
        [s,  c, ty],
        [0,  0,  1]
    ])

### Part 1: Forward Kinematics (FK)
Given a chain of bones (e.g., Shoulder -> Elbow -> Wrist), each bone has a local translation and rotation relative to its parent. 

To find the global position of any bone, we must multiply the parent's global transformation matrix by the child's local transformation matrix.

$$M_{global\_child} = M_{global\_parent} \times M_{local\_child}$$

In [ ]:
def compute_fk(bone_chain):
    """
    bone_chain: list of dictionaries, each containing 'local_tx', 'local_ty', 'local_angle'
    Returns a list of global 3x3 transformation matrices for each bone.
    """
    global_transforms = []
    current_transform = np.eye(3)
    
    for bone in bone_chain:
        local_tf = transform_2d(bone['local_tx'], bone['local_ty'], bone['local_angle'])
        
        # TODO: Update the current_transform by applying the local_tf
        current_transform = current_transform @ local_tf
        global_transforms.append(current_transform)
        
    return global_transforms

# Define a simple 2-bone arm (Upper Arm, Lower Arm)
arm_bones_animated = [
    {'local_tx': 0, 'local_ty': 0, 'local_angle': 45},   # Shoulder joint rotated 45 deg
    {'local_tx': 5, 'local_ty': 0, 'local_angle': -30}   # Elbow joint (5 units down the arm), rotated -30 deg
]

global_tfs = compute_fk(arm_bones_animated)
print("Global Transform Bone 0 (Shoulder):\n", np.round(global_tfs[0], 3))
print("\nGlobal Transform Bone 1 (Elbow):\n", np.round(global_tfs[1], 3))

### Part 2: Linear Blend Skinning (LBS)
When a model is rigged, its vertices are bound to the skeleton in a "Bind Pose" (usually a T-pose). 

To animate a vertex, we must:
1. Move it from global space back to the bone's local space using the **Inverse Bind Matrix** ($M_{bind}^{-1}$).
2. Move it to the new animated global position using the bone's current global transform ($M_{global}$).
3. Blend these positions based on the vertex weights ($w_i$).

$$v_{skin} = \sum_{i} w_i (M_{global\_i} M_{bind\_i}^{-1}) v_{rest}$$

In [ ]:
def linear_blend_skinning(vertex_rest, bone_transforms, bind_transforms, weights):
    """
    vertex_rest: [x, y, 1] array representing the vertex in bind pose
    bone_transforms: list of current animated global matrices
    bind_transforms: list of global matrices at bind pose
    weights: list of weights corresponding to each bone
    """
    v_skin = np.zeros(3)
    
    for i in range(len(weights)):
        if weights[i] > 0:
            # Compute Inverse Bind Matrix
            inv_bind = np.linalg.inv(bind_transforms[i])
            
            # Current animated bone transform
            bone_tf = bone_transforms[i]
            
            # TODO: Compute the vertex deformation for this bone and accumulate it
            v_skin += weights[i] * (bone_tf @ inv_bind @ vertex_rest)
            
    return v_skin

# 1. Define the Bind Pose (Straight arm, no rotation)
arm_bones_bind = [
    {'local_tx': 0, 'local_ty': 0, 'local_angle': 0},
    {'local_tx': 5, 'local_ty': 0, 'local_angle': 0}
]
bind_tfs = compute_fk(arm_bones_bind)

# 2. Define a vertex situated exactly at the elbow joint in bind pose (x=5, y=0)
v_rest = np.array([5.0, 0.0, 1.0])

# 3. Assign weights (50% influenced by shoulder, 50% influenced by elbow)
weights = [0.5, 0.5]

# 4. Calculate skinned position using the animated transforms from Part 1
v_animated = linear_blend_skinning(v_rest, global_tfs, bind_tfs, weights)

print(f"Original Vertex Position: {v_rest[:2]}")
print(f"Animated Skinned Position: {np.round(v_animated[:2], 3)}")

### Part 3: Artifact Verification
Ensure your `v_animated` prints out `[3.536, 3.536]`. If it does, your LBS matrix accumulation is functioning correctly. You are now prepared to map these mathematical principles onto node networks in OpenUSD.